In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///project.db")

In [6]:
## Top 10 rated movies
query = """
SELECT title, vote_average, vote_count, release_date
FROM movies
WHERE vote_count > 1000
ORDER BY vote_average DESC
LIMIT 10;
"""
pd.read_sql(query, engine)

,title,vote_average,vote_count,release_date
0,The Shawshank Redemption,8.5,8205,1994-09-23
1,The Godfather,8.4,5893,1972-03-14
2,Fight Club,8.3,9413,1999-10-15
3,Schindler's List,8.3,4329,1993-11-29
4,Spirited Away,8.3,3840,2001-07-20
5,The Godfather: Part II,8.3,3338,1974-12-20
6,Pulp Fiction,8.3,8428,1994-10-08
7,Whiplash,8.3,4254,2014-10-10
8,The Dark Knight,8.2,12002,2008-07-16
9,The Green Mile,8.2,4048,1999-12-10


In [ ]:
## Average revenue by genre
query = """
SELECT g.genre_name, ROUND(AVG(m.revenue), 0) AS avg_revenue
FROM movies m 
JOIN movie_genres mg ON m.id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
WHERE revenue > 0
GROUP BY g.genre_name
ORDER BY avg_revenue DESC;
"""
pd.read_sql(query, engine)

,genre_name,avg_revenue
0,Animation,225693025.0
1,Adventure,208660204.0
2,Fantasy,193354245.0
3,Family,162345495.0
4,Science Fiction,152456515.0
5,Action,141213098.0
6,War,84155874.0
7,Thriller,81044291.0
8,Mystery,78300927.0
9,Comedy,71289499.0


In [16]:
## Most prolific directors

query = """
SELECT cm.name, COUNT(*) AS movie_count
FROM movie_crew mc
JOIN crew_members cm ON mc.person_id = cm.person_id
WHERE mc.job = 'Director'
GROUP BY cm.name
HAVING COUNT(*) > 9
ORDER BY movie_count DESC;
"""
pd.read_sql(query, engine)

,name,movie_count
0,Steven Spielberg,27
1,Woody Allen,22
2,Martin Scorsese,21
3,Clint Eastwood,20
4,Robert Rodriguez,17
5,Spike Lee,16
6,Ridley Scott,16
7,Steven Soderbergh,15
8,Renny Harlin,15
9,Tim Burton,14


In [28]:
## Average movie budget by decade
query = """
SELECT (CAST(strftime('%Y', release_date) AS INT) / 10) * 10 AS decade, 
    ROUND(AVG(budget), 2) AS avg_budget,
    COUNT(*) AS movie_count
FROM movies m
WHERE budget > 0
GROUP BY decade
ORDER BY decade;
"""
pd.read_sql(query, engine)

,decade,avg_budget,movie_count
0,1910,385907.00,1
1,1920,31081333.33,3
2,1930,1438083.14,14
3,1940,2187238.10,21
4,1950,2877737.32,25
5,1960,5624457.70,63
6,1970,8583053.26,89
7,1980,15551411.74,232
8,1990,35860052.22,622
9,2000,40160280.81,1554


In [30]:
## Actors who've appeared in the most movies
query = """
SELECT cm.name, COUNT(*) AS movie_count
FROM movie_cast mc
JOIN cast_members cm ON mc.person_id = cm.person_id
GROUP BY cm.name
ORDER BY movie_count DESC
LIMIT 10;
"""
pd.read_sql(query, engine)

,name,movie_count
0,Samuel L. Jackson,67
1,Robert De Niro,57
2,Bruce Willis,51
3,Matt Damon,48
4,Morgan Freeman,46
5,Steve Buscemi,43
6,Liam Neeson,41
7,Owen Wilson,40
8,Johnny Depp,40
9,Nicolas Cage,39


## Advanced SQL queries BELOW

In [42]:
## Top 3 highest-grossing movies within each genre
query = """ 
WITH ranked_movies AS (
    SELECT g.genre_name, m.original_title AS title, m.revenue,
        ROW_NUMBER() OVER (
            PARTITION BY g.genre_name
            ORDER BY m.revenue DESC) AS rank
    FROM movie_genres mg 
    JOIN movies m ON mg.movie_id = m.id
    JOIN genres g ON mg.genre_id = g.genre_id
)

SELECT *
FROM ranked_movies
WHERE rank <= 3;
"""
pd.read_sql(query, engine)

,genre_name,title,revenue,rank
0,Action,Avatar,2787965087,1
1,Action,The Avengers,1519557910,2
2,Action,Jurassic World,1513528810,3
3,Adventure,Avatar,2787965087,1
4,Adventure,The Avengers,1519557910,2
5,Adventure,Jurassic World,1513528810,3
6,Animation,Frozen,1274219009,1
7,Animation,Minions,1156730962,2
8,Animation,Toy Story 3,1066969703,3
9,Comedy,Minions,1156730962,1


In [53]:
## Cumulative box office revenue over time
query = """
WITH revenue_by_year AS (
    SELECT CAST(strftime('%Y', release_date) AS INT) AS year, SUM(revenue) AS yearly_revenue
    FROM movies
    WHERE revenue > 0
    GROUP BY year
)

SELECT year, yearly_revenue,
    SUM(yearly_revenue) OVER (ORDER BY year) as cumulative_revenue
FROM revenue_by_year
WHERE year > (SELECT MAX(year) FROM revenue_by_year) - 25
ORDER BY year;
"""
pd.read_sql(query, engine)

,year,yearly_revenue,cumulative_revenue
0,1992,3756905404,3756905404
1,1993,3881952707,7638858111
2,1994,5928752504,13567610615
3,1995,6207143123,19774753738
4,1996,6808007511,26582761249
5,1997,9634746300,36217507549
6,1998,8223101034,44440608583
7,1999,10332740909,54773349492
8,2000,10952218337,65725567829
9,2001,13269869789,78995437618


In [65]:
## Directors ranked by average revenue per film (minimum 3 films)
query = """
SELECT cm.name, COUNT(*) AS film_count, ROUND(AVG(m.revenue), 1) AS avg_revenue
FROM movie_crew mc
JOIN crew_members cm ON mc.person_id = cm.person_id
JOIN movies m ON mc.movie_id = m.id
WHERE mc.job = 'Director' AND revenue > 0
GROUP BY cm.name
HAVING COUNT(*) >= 3
ORDER BY film_count DESC;
"""
pd.read_sql(query, engine)

,name,film_count,avg_revenue
0,Steven Spielberg,27,338792339.4
1,Clint Eastwood,20,125602944.4
2,Robert Rodriguez,17,65541151.8
3,Martin Scorsese,17,115096235.2
4,Ridley Scott,16,199347374.8
...,...,...,...
433,Alexandre Aja,3,75082772.3
434,Alex Kendrick,3,26052289.7
435,Alejandro Amenábar,3,31749275.7
436,Albert Hughes,3,86521956.7


In [89]:
## Movies with the best ROI with a minumum budget of $20k (return on investment)
query = """
WITH movies_budget_min AS (
    SELECT original_title AS title, budget, revenue
    FROM movies
    WHERE budget > 20000 
)

SELECT title, budget, revenue, ROUND(revenue * 1.0/budget, 2) AS roi
FROM movies_budget_min
ORDER BY roi DESC;
"""
pd.read_sql(query, engine)

,title,budget,revenue,roi
0,The Blair Witch Project,60000,248000000,4133.33
1,Super Size Me,65000,28575078,439.62
2,The Gallows,100000,42664410,426.64
3,Open Water,130000,54667954,420.52
4,The Texas Chain Saw Massacre,85000,30859000,363.05
...,...,...,...,...
3722,The Image Revolution,50000,0,0.00
3723,"Run, Hide, Die",50000,0,0.00
3724,Mutual Appreciation,500000,0,0.00
3725,Pink Narcissus,27000,0,0.00
